### Silver – vouchers

#### Purpose
Transform the Bronze `vouchers` table into a clean and analytics-ready
Silver table by:
- Standardizing column names using a reusable UDF
- Enforcing correct data types
- Handling nulls based on business rules
- Deduplicating records
- Isolating malformed records into a quarantine table

#### Source
- coffee.bronze.vouchers

#### Targets
- coffee.silver.vouchers
- coffee.silver.quarantine_vouchers


In [0]:
%python
# Widgets allow the same notebook to be executed across environments (DEV/PROD)
# and reused across multiple tables by changing only job parameters.
#
# default_watermark is used when the Silver table is empty (first run),
# enabling incremental ingestion logic without special casing.

dbutils.widgets.text("catalog", "coffee")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("source_table", "vouchers")
dbutils.widgets.text("default_watermark", "1900-01-01")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
source_table = dbutils.widgets.get("source_table")
default_watermark = dbutils.widgets.get("default_watermark")


In [0]:
%run ./Silver_utils/silver_transform_utils

In [0]:
%python
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{source_table}")


In [0]:
%python
# Standardize all incoming column names using the central UDF.
# This ensures consistent snake_case naming across all Silver tables,
# regardless of how the raw files were named in Bronze.

df_std = standardize_columns(df_bronze)


In [0]:
%python
df_std.createOrReplaceTempView(f"bronze_{source_table}_std")


In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table} (
  voucher_id INT,
  voucher_code STRING,
  discount_type STRING,
  discount_value DOUBLE,
  valid_from DATE,
  valid_to DATE,

  -- Bronze metadata
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP,
  load_dt DATE,
  source_file STRING,
  source_table STRING,

  -- Silver audit
  silver_loaded_at TIMESTAMP,
  silver_updated_at TIMESTAMP
)
USING DELTA
""")


In [0]:
-- Incremental extraction:
-- Only process Bronze rows that arrived after the latest loaded_at timestamp
-- already present in the Silver target table.
--
-- This prevents reprocessing old Bronze records and keeps Silver rerun-safe.

CREATE OR REPLACE TEMP VIEW bronze_vouchers_incremental AS
SELECT *
FROM bronze_vouchers_std
WHERE loaded_at >
(
  SELECT COALESCE(MAX(loaded_at), '1900-01-01')
  FROM coffee.silver.vouchers
);


In [0]:
%python

# Count invalid records for vouchers

# Required columns for vouchers:
# voucher_id, voucher_code, discount_type, discount_value, valid_from, valid_to

invalid_count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM bronze_vouchers_incremental
WHERE
  voucher_id IS NULL
  OR voucher_code IS NULL
  OR discount_type IS NULL
  OR discount_value IS NULL
  OR valid_from IS NULL
  OR valid_to IS NULL
""").collect()[0]["cnt"]

print("Invalid voucher rows:", invalid_count)


In [0]:
%python

#  Create and load vouchers quarantine table only if invalid rows exist


if invalid_count > 0:

    
    #  Create vouchers quarantine table
    
    spark.sql("""
    CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table}_quarantine (
      voucher_id STRING,
      voucher_code STRING,
      discount_type STRING,
      discount_value STRING,
      valid_from STRING,
      valid_to STRING,

      -- Bronze metadata
      loaded_at TIMESTAMP,
      updated_at TIMESTAMP,
      load_dt DATE,
      source STRING,
      source_file STRING,

      -- Quarantine metadata
      quarantine_reason STRING,
      quarantined_at TIMESTAMP
    )
    USING DELTA
    """)

    # 
    # Merge invalid rows into quarantine (idempotent)
    # 
    spark.sql("""
    MERGE INTO {catalog}.{silver_schema}.{source_table}_quarantine q
    USING (
      SELECT
        *,
        CASE
          WHEN voucher_id IS NULL THEN 'voucher_id is null'
          WHEN voucher_code IS NULL THEN 'voucher_code is null'
          WHEN discount_type IS NULL THEN 'discount_type is null'
          WHEN discount_value IS NULL THEN 'discount_value is null'
          WHEN valid_from IS NULL THEN 'valid_from is null'
          WHEN valid_to IS NULL THEN 'valid_to is null'
          ELSE 'unknown validation failure'
        END AS quarantine_reason,
        current_timestamp() AS quarantined_at
      FROM bronze_vouchers_incremental
      WHERE
        voucher_id IS NULL
        OR voucher_code IS NULL
        OR discount_type IS NULL
        OR discount_value IS NULL
        OR valid_from IS NULL
        OR valid_to IS NULL
    ) b
    ON q.voucher_id = b.voucher_id
    AND q.quarantine_reason = b.quarantine_reason
    WHEN NOT MATCHED THEN
    INSERT *;
    """)

else:
    print("No invalid voucher rows found. Quarantine table not created.")


In [0]:
%python
spark.sql(f""" 
MERGE INTO  {catalog}.{silver_schema}.{source_table} s
USING (

  
  -- Prepare clean, versioned voucher records
 
  SELECT
    TRY_CAST(voucher_id AS INT)          AS voucher_id,
    voucher_code,
    discount_type,
    TRY_CAST(discount_value AS DOUBLE)  AS discount_value,
    TRY_CAST(valid_from AS DATE)        AS valid_from,
    TRY_CAST(valid_to AS DATE)          AS valid_to,

    loaded_at,
    updated_at,
    load_dt,
    source_file,
    'coffee.bronze.vouchers' AS source_table,


    current_timestamp() AS silver_updated_at

  FROM (
   
    -- Deduplicate corrections for SAME validity period
    -- Deduplication logic:
-- Bronze may contain duplicates for the same business key.
-- We keep only the latest version of each record using:
--   ROW_NUMBER() OVER (PARTITION BY <business_key> ORDER BY updated_at DESC)
--
-- This ensures Silver contains a single clean record per business key.

    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY voucher_id
             ORDER BY updated_at DESC
           ) AS rn
    FROM bronze_vouchers_incremental
    WHERE
      voucher_id IS NOT NULL
      AND voucher_code IS NOT NULL
      AND discount_type IS NOT NULL
      AND discount_value IS NOT NULL
      AND valid_from IS NOT NULL
      AND valid_to IS NOT NULL
  )
  WHERE rn = 1

) b


-- Match on business grain (voucher_id + valid_from)

ON s.voucher_id = b.voucher_id
AND s.valid_from = b.valid_from

-- Existing validity period → UPDATE

WHEN MATCHED THEN
  UPDATE SET
    s.voucher_code        = b.voucher_code,
    s.discount_type       = b.discount_type,
    s.discount_value      = b.discount_value,
    s.valid_to            = b.valid_to,
    s.updated_at          = b.updated_at,
    s.load_dt             = b.load_dt,
    s.source_file         = b.source_file,
    s.silver_updated_at   = b.silver_updated_at


-- New validity period → INSERT

WHEN NOT MATCHED THEN
  INSERT (
    voucher_id,
    voucher_code,
    discount_type,
    discount_value,
    valid_from,
    valid_to,
    loaded_at,
    updated_at,
    load_dt,
    source_file,
    source_table,
    silver_loaded_at,
    silver_updated_at
  )
  VALUES (
    b.voucher_id,
    b.voucher_code,
    b.discount_type,
    b.discount_value,
    b.valid_from,
    b.valid_to,
    b.loaded_at,
    b.updated_at,
    b.load_dt,
    b.source_file,
    b.source_file,
    current_timestamp(),
    current_timestamp()
  )
  """)
